In [4]:
import psutil
import platform
from datetime import datetime

def get_system_info():
    return {
        "os": platform.system(),
        "os_version": platform.version(),
        "hostname": platform.node(),
        "boot_time": datetime.fromtimestamp(psutil.boot_time()).strftime("%Y-%m-%d %H:%M:%S")
    }

def get_cpu_info():
    return {
        "physical_cores": psutil.cpu_count(logical=False),
        "total_cores": psutil.cpu_count(logical=True),
        "cpu_usage_percent": psutil.cpu_percent(interval=1),
        "per_core_usage": psutil.cpu_percent(interval=1, percpu=True),
        "cpu_freq": psutil.cpu_freq()._asdict() if psutil.cpu_freq() else None,
        "load_avg": psutil.getloadavg() if hasattr(psutil, "getloadavg") else None
    }

def get_memory_info():
    mem = psutil.virtual_memory()
    swap = psutil.swap_memory()

    return {
        "total": mem.total,
        "available": mem.available,
        "used": mem.used,
        "percent": mem.percent,
        "swap_total": swap.total,
        "swap_used": swap.used,
        "swap_percent": swap.percent
    }

def get_disk_info():
    partitions = psutil.disk_partitions()
    disks = []

    for p in partitions:
        try:
            usage = psutil.disk_usage(p.mountpoint)
            disks.append({
                "device": p.device,
                "mountpoint": p.mountpoint,
                "fstype": p.fstype,
                "total": usage.total,
                "used": usage.used,
                "percent": usage.percent
            })
        except PermissionError:
            continue

    io = psutil.disk_io_counters()

    return {
        "partitions": disks,
        "io": io._asdict() if io else None
    }

def get_network_info():
    net = psutil.net_io_counters()

    return {
        "bytes_sent": net.bytes_sent,
        "bytes_recv": net.bytes_recv,
        "packets_sent": net.packets_sent,
        "packets_recv": net.packets_recv
    }

def get_process_info(limit=10):
    processes = []

    for p in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_percent']):
        try:
            processes.append(p.info)
        except:
            continue

    # CPU 사용률 높은 순 정렬
    processes = sorted(processes, key=lambda x: x['cpu_percent'], reverse=True)

    return processes[:limit]

def get_sensor_info():
    try:
        temps = psutil.sensors_temperatures()
        return temps
    except:
        return None

def get_battery_info():
    try:
        battery = psutil.sensors_battery()
        return battery._asdict() if battery else None
    except:
        return None

def collect_all():
    return {
        "system": get_system_info(),
        "cpu": get_cpu_info(),
        "memory": get_memory_info(),
        "disk": get_disk_info(),
        "network": get_network_info(),
        "top_processes": get_process_info(),
        "sensors": get_sensor_info(),
        "battery": get_battery_info()
    }

if __name__ == "__main__":
    import json
    data = collect_all()
    print(json.dumps(data, indent=2))

{
  "system": {
    "os": "Windows",
    "os_version": "10.0.19045",
    "hostname": "DESKTOP-9QEJN2F",
    "boot_time": "2026-04-17 14:38:07"
  },
  "cpu": {
    "physical_cores": 12,
    "total_cores": 20,
    "cpu_usage_percent": 3.4,
    "per_core_usage": [
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      18.2,
      0.0,
      1.5,
      0.0,
      3.0,
      0.0,
      0.0,
      0.0,
      13.6,
      10.8,
      9.2,
      4.6
    ],
    "cpu_freq": {
      "current": 2100.0,
      "min": 0.0,
      "max": 2100.0
    },
    "load_avg": [
      0.24,
      0.25,
      0.13
    ]
  },
  "memory": {
    "total": 68547305472,
    "available": 16231538688,
    "used": 52315766784,
    "percent": 76.3,
    "swap_total": 11811160064,
    "swap_used": 253214720,
    "swap_percent": 2.1
  },
  "disk": {
    "partitions": [
      {
        "device": "C:\\",
        "mountpoint": "C:\\",
        "fstype": "NTFS",
        "total": 511384711168,


In [14]:
import psutil
import time
from datetime import datetime

# 이전 값 저장 (속도 계산용)
prev_disk = psutil.disk_io_counters()
prev_net = psutil.net_io_counters()

def get_metrics():
    global prev_disk, prev_net

    # CPU
    cpu_percent = psutil.cpu_percent(interval=1)

    # Memory
    mem = psutil.virtual_memory()

    # Disk (속도 계산)
    disk = psutil.disk_io_counters()
    disk_read_speed = disk.read_bytes - prev_disk.read_bytes
    disk_write_speed = disk.write_bytes - prev_disk.write_bytes

    prev_disk = disk

    # Network (속도 계산)
    net = psutil.net_io_counters()
    net_recv_speed = net.bytes_recv - prev_net.bytes_recv
    net_send_speed = net.bytes_sent - prev_net.bytes_sent

    prev_net = net

    return {
        "time": datetime.now().strftime("%H:%M:%S"),

        "cpu_percent": cpu_percent,

        "memory_percent": mem.percent,
        "memory_used": mem.used,
        "memory_total": mem.total,

        "disk_read_Bps": disk_read_speed,
        "disk_write_Bps": disk_write_speed,

        "net_recv_Bps": net_recv_speed,
        "net_send_Bps": net_send_speed
    }

# 실시간 루프
if __name__ == "__main__":
    print("Start monitoring...\n")

    while True:
        data = get_metrics()

        print(
            f"[{data['time']}] "
            f"CPU: {data['cpu_percent']}% | "
            f"MEM: {data['memory_percent']}% | "
            # f"DISK R/W: {data['disk_read_Bps']}/{data['disk_write_Bps']} B/s | "
            # f"NET R/S: {data['net_recv_Bps']}/{data['net_send_Bps']} B/s"
        )

        time.sleep(1)

Start monitoring...

[15:36:38] CPU: 3.9% | MEM: 39.6% | 
[15:36:40] CPU: 2.7% | MEM: 39.6% | 
[15:36:42] CPU: 3.2% | MEM: 39.6% | 
[15:36:44] CPU: 1.9% | MEM: 39.6% | 
[15:36:46] CPU: 5.1% | MEM: 39.6% | 
[15:36:48] CPU: 5.9% | MEM: 39.6% | 
[15:36:50] CPU: 2.9% | MEM: 39.6% | 
[15:36:52] CPU: 3.9% | MEM: 39.6% | 
[15:36:54] CPU: 4.1% | MEM: 39.6% | 
[15:36:56] CPU: 6.5% | MEM: 39.6% | 
[15:36:58] CPU: 4.6% | MEM: 39.6% | 
[15:37:00] CPU: 3.6% | MEM: 39.6% | 
[15:37:02] CPU: 2.5% | MEM: 39.6% | 
[15:37:04] CPU: 3.9% | MEM: 39.6% | 
[15:37:06] CPU: 2.4% | MEM: 39.6% | 
[15:37:08] CPU: 3.1% | MEM: 39.6% | 
[15:37:10] CPU: 5.3% | MEM: 39.6% | 
[15:37:12] CPU: 3.2% | MEM: 39.6% | 
[15:37:14] CPU: 6.5% | MEM: 39.6% | 


KeyboardInterrupt: 